# MAP Competition - Qwen3-8B Inference with Multi-Step Explainable AI

This notebook uses the **same trained Qwen3-8B model** for both:
1. **Classification** - Predicting misconception labels
2. **Explanation** - Multi-step diagnosis and feedback generation

No Knowledge Base cards needed - the model diagnoses directly using a structured pipeline.

In [1]:
# ============================================================================
# CELL 1: Setup and Install Dependencies
# ============================================================================

import os
import sys
import subprocess

# Install bitsandbytes from local wheel FIRST (before any other imports)
BNB_WHEEL = "/kaggle/input/datasets/mdmarufhasanrubab/bnb-wheel/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl"
if os.path.exists(BNB_WHEEL):
    print("Installing bitsandbytes from local wheel...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", BNB_WHEEL, "-q"], check=True)
    print("bitsandbytes installed successfully!", flush=True)
else:
    print(f"WARNING: bitsandbytes wheel not found at {BNB_WHEEL}", flush=True)
    # Try alternate path
    alt_path = "/kaggle/input/bnb-wheel/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl"
    if os.path.exists(alt_path):
        print(f"Found at alternate path: {alt_path}", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", alt_path, "-q"], check=True)
        print("bitsandbytes installed successfully!", flush=True)

Installing bitsandbytes from local wheel...
bitsandbytes installed successfully!


In [2]:
# ============================================================================
# CELL 2: Imports
# ============================================================================

import gc
import json
import re
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def log(msg):
    print(msg, flush=True)

log("Starting imports...")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import PeftModel
from scipy.special import softmax
from tqdm import tqdm

log("Imports complete!")

# Check GPU
if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)}")
    log(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    log("WARNING: No GPU available!")

device = "cuda:0" if torch.cuda.is_available() else "cpu"

Starting imports...
Imports complete!
GPU: Tesla T4
GPU Memory: 15.6 GB


In [3]:
# ============================================================================
# CELL 3: Configuration
# ============================================================================

# Data paths
TRAIN_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
TEST_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"

# Qwen3-8B paths (base model saved with 4-bit quantization)
BASE_MODEL_PATH = "/kaggle/input/datasets/mkhanshovon43/base-modelsh7bq2-5-7b3-8b/loaded_models/Qwen3-8B"
ADAPTER_PATH = "/kaggle/input/datasets/mdmarufhasanrubab/map-qwen3-8b-enhanced"

# Model settings
MAX_LEN = 256
BATCH_SIZE = 4

# Explainer settings - samples from TRAINING data for demo
EXPLAIN_SAMPLES = 5  # Reduced for speed
MAX_NEW_TOKENS = 250  # Reduced for speed

log("Configuration loaded.")

Configuration loaded.


In [4]:
# ============================================================================
# CELL 4: Helper Functions
# ============================================================================

import glob

def find_model_path(base_path):
    """Find the actual model directory within a dataset."""
    if not os.path.exists(base_path):
        log(f"Path does not exist: {base_path}")
        return base_path

    # Check base path first
    if os.path.exists(os.path.join(base_path, "config.json")):
        has_weights = (os.path.exists(os.path.join(base_path, "model.safetensors")) or
                      os.path.exists(os.path.join(base_path, "pytorch_model.bin")) or
                      glob.glob(os.path.join(base_path, "model*.safetensors")))
        if has_weights:
            return base_path

    # Search for config.json with model weights
    for depth in range(1, 6):
        pattern = os.path.join(base_path, "/".join(["*"] * depth), "config.json")
        matches = glob.glob(pattern)
        for m in matches:
            dir_path = os.path.dirname(m)
            has_weights = (os.path.exists(os.path.join(dir_path, "model.safetensors")) or
                          os.path.exists(os.path.join(dir_path, "pytorch_model.bin")) or
                          glob.glob(os.path.join(dir_path, "model*.safetensors")))
            if has_weights:
                return dir_path

    return base_path


def find_adapter_path(base_path):
    """Find the actual adapter directory within a dataset."""
    if not os.path.exists(base_path):
        log(f"Path does not exist: {base_path}")
        return base_path

    # Check base path first
    if os.path.exists(os.path.join(base_path, "adapter_config.json")):
        return base_path

    # Search for adapter_config.json
    for depth in range(1, 7):
        pattern = os.path.join(base_path, "/".join(["*"] * depth), "adapter_config.json")
        matches = glob.glob(pattern)
        if matches:
            # Prefer paths with "adapter" or "full_train" in them
            preferred = [m for m in matches if "adapter" in m.lower() or "full_train" in m.lower()]
            if preferred:
                return os.path.dirname(preferred[0])
            return os.path.dirname(matches[0])

    return base_path


def format_input(row):
    """Format input text for classification."""
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


def clear_gpu_memory():
    """Clear GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

log("Helper functions defined.")

Helper functions defined.


In [5]:
# ============================================================================
# CELL 5: Load and Prepare Data
# ============================================================================

log("Loading data...")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

log(f"Train: {train_df.shape}, Test: {test_df.shape}")

# Prepare labels
train_df['Misconception'] = train_df['Misconception'].fillna('NA')
train_df['target'] = train_df['Category'] + ':' + train_df['Misconception']

le = LabelEncoder()
le.fit(train_df['target'])
n_classes = len(le.classes_)
log(f"Number of classes: {n_classes}")

# Find correct answers for is_correct flag
idx = train_df['Category'].apply(lambda c: c.split('_')[0]) == 'True'
correct = train_df.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
correct = correct[['QuestionId', 'MC_Answer']]
correct['is_correct'] = 1

# Build correct answer lookup
correct_by_q = dict(zip(correct['QuestionId'], correct['MC_Answer']))

# Merge is_correct into test
test_df = test_df.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
test_df['is_correct'] = test_df['is_correct'].fillna(0)

# Format input text
test_df['text'] = test_df.apply(format_input, axis=1)

# Build family map for filtering
fam_map = {}
for _, row in test_df.iterrows():
    fam_map[row['row_id']] = 'True_' if row['is_correct'] == 1 else 'False_'

log("Data prepared.")

Loading data...
Train: (36696, 7), Test: (3, 5)
Number of classes: 65
Data prepared.


In [6]:
# ============================================================================
# CELL 6: Find Model Paths
# ============================================================================

log("\nSearching for model paths...")

# List available inputs for debugging
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    log(f"Available inputs: {os.listdir(input_dir)}")
    for d in os.listdir(input_dir):
        subpath = os.path.join(input_dir, d)
        if os.path.isdir(subpath):
            try:
                log(f"  {d}/: {os.listdir(subpath)[:5]}")
            except:
                pass

# Find base model
base_model_path = find_model_path(BASE_MODEL_PATH)
log(f"\nBase model path: {base_model_path}")

# Find adapter
adapter_path = find_adapter_path(ADAPTER_PATH)
log(f"Adapter path: {adapter_path}")

# Verify paths
if not os.path.exists(base_model_path):
    raise FileNotFoundError(f"Base model not found: {base_model_path}")
if not os.path.exists(adapter_path):
    raise FileNotFoundError(f"Adapter not found: {adapter_path}")

log("\nPaths verified!")


Searching for model paths...
Available inputs: ['datasets', 'competitions']
  datasets/: ['mdmarufhasanrubab', 'mkhanshovon43', 'heisamit3', 'ahmadmuhammad26']
  competitions/: ['map-charting-student-math-misunderstandings']

Base model path: /kaggle/input/datasets/mkhanshovon43/base-modelsh7bq2-5-7b3-8b/loaded_models/Qwen3-8B
Adapter path: /kaggle/input/datasets/mdmarufhasanrubab/map-qwen3-8b-enhanced/qwen3-8b-qlora/adapter

Paths verified!


In [7]:
# ============================================================================
# CELL 7: Load Classification Model (Qwen3-8B + LoRA)
# ============================================================================

log("\n" + "="*60)
log("Loading Qwen3-8B Classification Model")
log("="*60)

# Load tokenizer
log("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
log("Tokenizer loaded.")

# Load base model (saved with 4-bit quantization)
log("Loading base model with 4-bit quantization...")
classification_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=n_classes,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)
log("Base model loaded.")

# Resize embeddings
classification_model.resize_token_embeddings(len(tokenizer))

# Load LoRA adapter
log("Loading LoRA adapter...")
classification_model = PeftModel.from_pretrained(
    classification_model,
    adapter_path,
    local_files_only=True
)
classification_model.config.pad_token_id = tokenizer.pad_token_id
classification_model.eval()
log("Classification model ready!")


Loading Qwen3-8B Classification Model
Loading tokenizer...
Tokenizer loaded.
Loading base model with 4-bit quantization...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/mkhanshovon43/base-modelsh7bq2-5-7b3-8b/loaded_models/Qwen3-8B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model loaded.
Loading LoRA adapter...
Classification model ready!


In [8]:
# ============================================================================
# CELL 8: Run Classification Inference
# ============================================================================

log("\n" + "="*60)
log("Running Classification Inference")
log("="*60)

# Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds_test = Dataset.from_pandas(test_df[['text']])
ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
dataloader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

# Run inference
log(f"Running inference on {len(ds_test)} samples...")
all_logits = []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Classification"):
        batch = {k: v.to(classification_model.device) for k, v in batch.items()}
        outputs = classification_model(**batch)
        all_logits.append(outputs.logits.float().cpu().numpy())

predictions = np.concatenate(all_logits, axis=0)
probs = softmax(predictions, axis=1)
top_indices = np.argsort(-probs, axis=1)

# Decode labels
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)

# Store predictions in test_df for later use
test_df['pred_label'] = [top_labels[i, 0] for i in range(len(test_df))]
test_df['pred_top3'] = [' '.join(top_labels[i, :3]) for i in range(len(test_df))]
test_df['pred_probs'] = [probs[i, top_indices[i, :3]].tolist() for i in range(len(test_df))]

log("Classification complete!")
log(f"\nSample predictions:")
log(test_df[['row_id', 'pred_label', 'pred_top3']].head())


Running Classification Inference


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Running inference on 3 samples...


Classification: 100%|██████████| 1/1 [00:02<00:00,  2.48s/it]

Classification complete!

Sample predictions:
   row_id               pred_label  \
0   36696          True_Correct:NA   
1   36697  False_Misconception:WNB   
2   36698          True_Neither:NA   

                                           pred_top3  
0   True_Correct:NA True_Neither:NA False_Neither:NA  
1  False_Misconception:WNB False_Neither:NA False...  
2  True_Neither:NA False_Neither:NA True_Misconce...  


In [9]:
# ============================================================================
# CELL 9: Create Submission with Family Filtering
# ============================================================================

log("\nCreating submission with family filtering...")

final_predictions = []

for i, row_id in enumerate(test_df.row_id.values):
    pref = fam_map.get(row_id, 'False_')

    # Filter predictions by family prefix
    filtered = [lbl for lbl in top_labels[i] if lbl.startswith(pref)]

    # Take top 3
    top3 = filtered[:3]

    # Fill if needed
    while len(top3) < 3:
        filler = f"{pref}Neither:NA"
        if filler not in top3:
            top3.append(filler)
        else:
            top3.append(f"{pref}Correct:NA" if pref == "True_" else f"{pref}Misconception:NA")

    final_predictions.append(" ".join(top3[:3]))

# Save submission
submission = pd.DataFrame({
    'row_id': test_df.row_id.values,
    'Category:Misconception': final_predictions
})
submission.to_csv('submission.csv', index=False)

log("\n" + "="*60)
log("SUBMISSION SAVED: submission.csv")
log("="*60)
log(str(submission.head(10)))
log(f"\nTotal predictions: {len(submission)}")


Creating submission with family filtering...

SUBMISSION SAVED: submission.csv
   row_id                             Category:Misconception
0   36696  True_Correct:NA True_Neither:NA True_Misconcep...
1   36697  False_Misconception:WNB False_Neither:NA False...
2   36698  True_Neither:NA True_Misconception:Shorter_is_...

Total predictions: 3


In [10]:
# ============================================================================
# CELL 10: Free Classification Model, Load CausalLM for Explanation
# ============================================================================

log("\n" + "="*60)
log("Loading Qwen3-8B for Explanation Generation")
log("="*60)

# Free classification model
log("Freeing classification model memory...")
del classification_model
clear_gpu_memory()
log("Memory cleared.")

# Load the same Qwen3-8B base as CausalLM for text generation
log("Loading Qwen3-8B as CausalLM...")
explainer_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)
explainer_model.eval()
log("Explainer model loaded!")


Loading Qwen3-8B for Explanation Generation
Freeing classification model memory...
Memory cleared.
Loading Qwen3-8B as CausalLM...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Explainer model loaded!


In [11]:
# ============================================================================
# CELL 11: Simplified Explanation Pipeline (Faster - Single Step)
# ============================================================================

def llm_generate(prompt: str, max_new_tokens: int = 250) -> str:
    """Generate text using Qwen3-8B."""
    messages = [{"role": "user", "content": prompt}]
    
    try:
        chat_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        chat_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    
    inputs = tokenizer(chat_prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(explainer_model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = explainer_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, 
            num_beams=1, pad_token_id=tokenizer.eos_token_id
        )
    
    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    input_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    return full_text[len(input_text):].strip() if full_text.startswith(input_text) else full_text.strip()


def explain_student_work(row) -> dict:
    """
    Single-step comprehensive explanation (faster than multi-step).
    Combines diagnosis and feedback generation.
    """
    qtext = str(row["QuestionText"])[:400]
    ans = str(row["MC_Answer"])
    expl = str(row["StudentExplanation"])[:400]
    true_label = str(row.get("target", row.get("label_str", "")))
    correct_ans = str(correct_by_q.get(row.get("QuestionId", ""), ""))
    
    result = {
        "question": qtext,
        "student_answer": ans,
        "student_explanation": expl,
        "true_label": true_label,
        "correct_answer": correct_ans,
        "feedback": ""
    }
    
    # Single comprehensive prompt
    prompt = f"""You are a kind math teacher analyzing a student's work.

QUESTION: {qtext}
STUDENT'S ANSWER: {ans}
CORRECT ANSWER: {correct_ans if correct_ans else "(derive from question)"}
STUDENT'S EXPLANATION: "{expl}"
LABELED MISCONCEPTION: {true_label}

TASK: Analyze and provide feedback in this format:

1. SOLVE: Show the correct solution briefly (2-3 lines)
2. DIAGNOSIS: 
   - What error type? (CALCULATION/CONCEPTUAL/INCOMPLETE/CORRECT)
   - Quote the specific mistake from student's explanation
   - What should it actually be?
3. FEEDBACK: Write 4-6 lines of encouraging teacher feedback that:
   - Acknowledges what the student did right (if anything)
   - Explains exactly where they went wrong
   - Shows the correct approach
   - If CALCULATION error: praise the method, fix only the arithmetic
   - If CONCEPTUAL error: teach the right concept simply

Be concise and specific."""

    feedback = llm_generate(prompt, MAX_NEW_TOKENS)
    result["feedback"] = feedback
    
    return result

log("Explanation function defined (single-step for speed).")

Explanation function defined (single-step for speed).


In [12]:
# ============================================================================
# CELL 12: Generate Explanations for TRAINING Samples (Demo)
# ============================================================================

log("\n" + "="*80)
log("EXPLAINABLE AI — DEMO ON TRAINING SAMPLES")
log("="*80)

# Prepare training data for explanation demo
train_df['is_correct'] = train_df['Category'].str.startswith('True').astype(int)

# Sample diverse cases from training set (mix of correct and incorrect)
rng = np.random.default_rng(42)
incorrect_samples = train_df[train_df['is_correct'] == 0].sample(n=min(4, len(train_df)), random_state=42)
correct_samples = train_df[train_df['is_correct'] == 1].sample(n=min(1, len(train_df)), random_state=42)
demo_samples = pd.concat([incorrect_samples, correct_samples]).head(EXPLAIN_SAMPLES)

log(f"Generating explanations for {len(demo_samples)} training samples...")

explanations = []

for idx, (_, row) in enumerate(demo_samples.iterrows()):
    log(f"\n{'─'*80}")
    log(f"SAMPLE {idx + 1} / {len(demo_samples)}")
    log(f"{'─'*80}")
    log(f"Question: {str(row['QuestionText'])[:150]}")
    log(f"Student Answer: {row['MC_Answer']}")
    log(f"Student Explanation: {str(row['StudentExplanation'])[:150]}")
    log(f"True Label: {row['target']}")
    
    try:
        result = explain_student_work(row)
        explanations.append(result)
        log(f"\n--- MODEL FEEDBACK ---")
        log(result['feedback'])
    except Exception as e:
        log(f"ERROR: {e}")
        explanations.append({"error": str(e)})

log(f"\n\nGenerated {len(explanations)} explanations.")


EXPLAINABLE AI — DEMO ON TRAINING SAMPLES
Generating explanations for 5 training samples...

────────────────────────────────────────────────────────────────────────────────
SAMPLE 1 / 5
────────────────────────────────────────────────────────────────────────────────
Question: What number belongs in the box?
\(
(-8)-(-5)=
\square\)
Student Answer: \( 3 \)
Student Explanation: because if there is 2 negative numbers then that makes a positive answer
True Label: False_Misconception:Positive


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- MODEL FEEDBACK ---
1. **SOLVE**:  
   $(-8) - (-5) = -8 + 5 = -3$

2. **DIAGNOSIS**:  
   - **Error Type**: CONCEPTUAL  
   - **Mistake**: "because if there is 2 negative numbers then that makes a positive answer"  
   - **What it should be**: Subtracting a negative is the same as adding a positive, but the sign of the result depends on the values. Here, $-8$ is larger in magnitude than $5$, so the result is negative.

3. **FEEDBACK**:  
   You're on the right track by recognizing that subtracting a negative is like adding a positive. However, the key is to compare the magnitudes of the numbers. Since $-8$ is larger than $5$, the result should be negative. The correct answer is $-3$. Keep up the good work!

────────────────────────────────────────────────────────────────────────────────
SAMPLE 2 / 5
────────────────────────────────────────────────────────────────────────────────
Question: It takes \( 3 \) people a total of \( 192 \) hours to build a wall.

How long would it take i

In [13]:
# ============================================================================
# CELL 13: Save Explanations to JSON
# ============================================================================

# Save explanations
explanations_path = "explanations.json"
with open(explanations_path, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False, indent=2)

log(f"\nExplanations saved to: {explanations_path}")

# Summary statistics
error_types = [e.get('error_type', 'unknown') for e in explanations if 'error_type' in e]
if error_types:
    log("\nError Type Distribution:")
    from collections import Counter
    for etype, count in Counter(error_types).items():
        log(f"  {etype}: {count}")


Explanations saved to: explanations.json


In [14]:
# ============================================================================
# CELL 14: Cleanup
# ============================================================================

log("\nCleaning up...")

del explainer_model
del tokenizer
clear_gpu_memory()

log("\n" + "="*60)
log("NOTEBOOK COMPLETE!")
log("="*60)
log(f"\nOutputs:")
log(f"  - submission.csv: Competition submission")
log(f"  - explanations.json: Multi-step explanations for {len(explanations)} samples")


Cleaning up...

NOTEBOOK COMPLETE!

Outputs:
  - submission.csv: Competition submission
  - explanations.json: Multi-step explanations for 5 samples
